#Atividade Prática: Visualização Geoespacial com Folium

In [2]:
import pandas as pd
import numpy as np
import folium

from folium import plugins
from folium.plugins import MarkerCluster, HeatMap, MiniMap

In [3]:
# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

In [4]:
df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

#Parte 01
##Inicialização e Marcadores Básicos

1. Crie um mapa base (folium.Map) centralizado na coordenada média de todos os imóveis do DataFrame. Ajuste o nível de zoom inicial (zoom_start) para 12 e utilize o estilo de mapa padrão (OpenStreetMap).<br>


In [11]:
mapa_base = folium.Map(location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()], zoom_start=12, tiles='OpenStreetMap')
display(mapa_base)

2. Itere sobre as 5 primeiras linhas do DataFrame e adicione um marcador simples (folium.Marker) para cada imóvel. Configure o popup para exibir o tipo do imóvel e o valor de venda.<br>


In [13]:
mapa_makers = folium.Map(
    location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, linha in df_mapa.head(5).iterrows():
    popup_texto = f"{linha['tipo']} — R$ {linha['valor_venda']:,.2f}"
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=popup_texto
    ).add_to(mapa_makers)

display(mapa_makers)

#Parte 02
##Customização Visual com Marcadores Circulares

3. Crie um novo mapa base. Desta vez, utilize folium.CircleMarker para representar todos os imóveis da base.<br>


In [14]:
cores = {'Nova Iguaçu': 'blue', 'Queimados': 'orange'}

mapa_circulos = folium.Map(
    location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, linha in df_mapa.iterrows():
    folium.CircleMarker(
        location=[linha['latitude'], linha['longitude']],
        radius=8,
        color=cores[linha['cidade']],
        fill=True,
        fill_color=cores[linha['cidade']],
        fill_opacity=0.7,
        tooltip="Clique para detalhes"
    ).add_to(mapa_circulos)

display(mapa_circulos)

4. Configure os marcadores circulares com as seguintes regras de negócio:
- Raio: Fixo em 8 pixels.
- Cor de preenchimento: Azul para 'Nova Iguaçu' e Laranja para 'Queimados'.
- Tooltip: Exiba o texto "Clique para detalhes" ao passar o mouse.<br>



#Parte 03
##Agrupamento Inteligente (Clustering)

5. Quando há muitos pontos próximos, o mapa pode ficar poluído. Crie um terceiro mapa e instancie um objeto MarkerCluster().<br>


In [17]:
cores_tipo = {'Casa': 'green', 'Apartamento': 'blue', 'Terreno': 'gray'}

mapa_cluster = folium.Map(
    location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, linha in df_mapa.iterrows():
    popup_texto = f"{linha['tipo']} — R$ {linha['valor_venda']:,.2f}"
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=popup_texto,
        tooltip="Clique para detalhes",
        icon=folium.Icon(color=cores_tipo[linha['tipo']], prefix='fa', icon='home')
    ).add_to(mapa_cluster)

mapa_cluster.save('mapa_imoveis_baixada.html')
print("Mapa salvo como 'mapa_imoveis_baixada.html'")
display(mapa_cluster)

Mapa salvo como 'mapa_imoveis_baixada.html'


6. Adicione todos os imóveis do DataFrame a esse cluster (em vez de adicioná-los diretamente ao mapa base). Utilize ícones customizados (folium.Icon) variando a cor conforme o tipo de imóvel (ex: verde para Casa, azul para Apartamento, cinza para Terreno).<br>


7. Salve o mapa final em um arquivo HTML chamado mapa_imoveis_baixada.html.<br>
